In [101]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith(".csv"):
            print(os.path.join(root, file))

/kaggle/input/datasets/mohammadmusfiq/bbc-text/bbc-text-preprocessed.csv


DATASET 2 - BBC TEXT

In [102]:
import os
import glob
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

print("Imported required libraries...")

Imported required libraries...


1. CONFIGURATION

In [103]:
# DistilBERT configuration
MODEL_NAME = "distilbert-base-uncased"

LEARNING_RATE = 0.00003
BATCH_SIZE = 32
WEIGHT_DECAY = 0.01

EPOCHS = 5
WARMUP_RATIO = 0.1
MAX_LEN = 128
DROPOUT = 0.1

SEED = 42

print("Defined Parameters...")

Defined Parameters...


2. DATASET PATH

In [104]:

FILE_NAME = "bbc-text-preprocessed.csv"


file_matches = glob.glob(
    f"/kaggle/input/**/{FILE_NAME}",
    recursive=True
)

if len(file_matches) == 0:
    raise FileNotFoundError(
        f"Could not find {FILE_NAME} inside /kaggle/input/"
    )

NEWS_FILE = file_matches[0]

print("Dataset path:", NEWS_FILE)



Dataset path: /kaggle/input/datasets/mohammadmusfiq/bbc-text/bbc-text-preprocessed.csv


3. DEVICE

In [105]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


Device: cuda
GPU: Tesla T4


4. SEED

In [106]:

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)



5. DATASET CLASS

In [107]:
class TextDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_len=MAX_LEN
    ):

        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_len
        )

        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(
                value[idx],
                dtype=torch.long
            )
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item
        
print("Dataset class defined...")


Dataset class defined...


6. METRICS

In [108]:

def compute_metrics(pred):

    labels = pred.label_ids
    predictions = pred.predictions

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    preds = np.argmax(
        predictions,
        axis=-1
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="weighted",
            zero_division=0
        )
    )

    accuracy = accuracy_score(
        labels,
        preds
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("Metrics class defined...")

Metrics class defined...


7. LOAD DATASET

In [109]:
print("\n" + "=" * 70)
print("Running DistilBERT on: Dataset2_BBCTEXT")
print("=" * 70)

df = pd.read_csv(NEWS_FILE)

print("Original shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())



Running DistilBERT on: Dataset2_BBCTEXT
Original shape: (2225, 2)

Columns:
['category', 'clean_text']


8. DATA CLEANING

In [110]:

required_columns = [
    "clean_text",
    "category"
]

missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )



df = df.dropna(
    subset=[
        "clean_text",
        "category"
    ]
).copy()



df["clean_text"] = (
    df["clean_text"]
    .astype(str)
    .str.strip()
)

df["category"] = (
    df["category"]
    .astype(str)
    .str.strip()
    .str.lower()
)



df = df[
    (df["clean_text"] != "") &
    (df["category"] != "")
].reset_index(drop=True)

print(
    "After removing missing/empty values:",
    df.shape
)



before_duplicates = len(df)

df = df.drop_duplicates(
    subset=["clean_text"]
).reset_index(drop=True)

removed_duplicates = (
    before_duplicates - len(df)
)

print(
    "Duplicate rows removed:",
    removed_duplicates
)

print(
    "Final dataset shape:",
    df.shape
)



After removing missing/empty values: (2225, 2)
Duplicate rows removed: 108
Final dataset shape: (2117, 2)


LABEL ENCODING

In [111]:
labels_unique = sorted(
    df["category"].unique()
)

label2id = {
    label: idx
    for idx, label in enumerate(labels_unique)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

df["label_id"] = df["category"].map(
    label2id
)

num_labels = len(labels_unique)

print(
    "\nNumber of classes:",
    num_labels
)

print(
    "Classes:",
    labels_unique
)

print(
    "\nClass distribution:"
)

print(
    df["category"].value_counts()
)

print(
    "\nLabel mapping:",
    label2id
)



Number of classes: 5
Classes: ['business', 'entertainment', 'politics', 'sport', 'tech']

Class distribution:
category
business         503
sport            501
politics         403
entertainment    369
tech             341
Name: count, dtype: int64

Label mapping: {'business': 0, 'entertainment': 1, 'politics': 2, 'sport': 3, 'tech': 4}


10. TRAIN / VALIDATION / TEST SPLIT

In [112]:
train_texts, temp_texts, train_labels, temp_labels = (
    train_test_split(
        df["clean_text"].tolist(),
        df["label_id"].tolist(),
        test_size=0.20,
        random_state=SEED,
        stratify=df["label_id"]
    )
)

val_texts, test_texts, val_labels, test_labels = (
    train_test_split(
        temp_texts,
        temp_labels,
        test_size=0.50,
        random_state=SEED,
        stratify=temp_labels
    )
)


print(
    "\nTraining samples:",
    len(train_texts)
)

print(
    "Validation samples:",
    len(val_texts)
)

print(
    "Testing samples:",
    len(test_texts)
)


Training samples: 1693
Validation samples: 212
Testing samples: 212


11. TOKENIZER

In [113]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)


12. CREATE DATASETS

In [114]:
train_dataset = TextDataset(
    train_texts,
    train_labels,
    tokenizer,
    MAX_LEN
)

val_dataset = TextDataset(
    val_texts,
    val_labels,
    tokenizer,
    MAX_LEN
)

test_dataset = TextDataset(
    test_texts,
    test_labels,
    tokenizer,
    MAX_LEN
)

print("Dataset object created...")


Dataset object created...


13. DistilBERT MODEL

In [115]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    dropout=DROPOUT,
    attention_dropout=DROPOUT
)

model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


14. TRAINING ARGUMENTS

In [116]:
training_args = TrainingArguments(

    output_dir=f"/kaggle/working/Dataset2_BBCTEXT_DistilBERT_LR{LEARNING_RATE}_BS{BATCH_SIZE}_WD{WEIGHT_DECAY}",

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,

    warmup_ratio=WARMUP_RATIO,

    eval_strategy="epoch",

    save_strategy="epoch",
    save_total_limit=1,

    load_best_model_at_end=False,

    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=20,

    optim="adamw_torch",

    seed=SEED,
    data_seed=SEED,

    report_to="none",

    fp16=torch.cuda.is_available(),

    dataloader_num_workers=2
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


15. TRAINER

In [117]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

 
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print("Trainer...")

Trainer...


16. TRAINING

In [118]:
print("\nStarting training...")

train_result = trainer.train()



Starting training...


Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,3.047461,1.624261,0.952830,0.958686,0.952830,0.953077
2,1.552071,0.411056,0.976415,0.976347,0.976415,0.976327
3,0.278641,0.244308,0.981132,0.981256,0.981132,0.981128
4,0.200122,0.243034,0.971698,0.972548,0.971698,0.971680
5,0.113115,0.251142,0.971698,0.972548,0.971698,0.971680


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

17. FINAL TEST EVALUATION

In [119]:
print("\nEvaluating on unseen test set...")

test_metrics = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test"
)


Evaluating on unseen test set...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


early stopping required metric_for_best_model, but did not find eval_f1 so early stopping is disabled


18. FINAL RESULTS

In [120]:
print("\n" + "=" * 70)
print("DATASET 2 - BBC TEXT - DistilBERT RESULTS")
print("=" * 70)

print(
    f"Learning Rate: {LEARNING_RATE} | "
    f"Batch Size: {BATCH_SIZE} | "
    f"Weight Decay: {WEIGHT_DECAY}"
)

print(
    f"Accuracy  : "
    f"{test_metrics['test_accuracy'] * 100:.2f}%"
)

print(
    f"Precision : "
    f"{test_metrics['test_precision'] * 100:.2f}%"
)

print(
    f"Recall    : "
    f"{test_metrics['test_recall'] * 100:.2f}%"
)

print(
    f"F1 Score  : "
    f"{test_metrics['test_f1'] * 100:.2f}%"
)

print("=" * 70)



DATASET 2 - BBC TEXT - DistilBERT RESULTS
Learning Rate: 3e-05 | Batch Size: 32 | Weight Decay: 0.01
Accuracy  : 98.58%
Precision : 98.58%
Recall    : 98.58%
F1 Score  : 98.58%
